# Adult Income Data Cleaning & Feature Engineering

## 1. Project Overview

This project performs data cleaning and feature engineering on the
UCI Adult Income dataset.

The objective is to prepare a clean, analysis-ready dataset for studying
which demographic, educational, and employment characteristics are
associated with an annual income above USD 50K.

This project covers the data preparation workflow:

- Data collection
- Raw data inspection
- Data cleaning
- Missing-value handling
- Data validation
- Feature engineering
- Final data quality audit
- Cleaned dataset export

### Business Question

> Which demographic, education, and work characteristics are associated with
> an annual income above USD 50K in this census dataset?

### Analysis Approach

This is a **descriptive analysis**. The results identify patterns and
associations within the dataset but do not establish causal relationships.

### Dataset Source

**UCI Machine Learning Repository — Adult Dataset**

The analysis uses the official:

- `adult.data`
- `adult.test`

files provided by the UCI Machine Learning Repository.

## 2. Import Libraries

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import requests

print("Libraries imported successfully!")

Libraries imported successfully!


## 3. Data Collection

The Adult Income dataset is obtained from the UCI Machine Learning Repository.

The original dataset is provided in two files:

- `adult.data` — training data
- `adult.test` — test data

Both files are downloaded from the official UCI source and stored in the
`data/raw/` directory without modifying their original contents.

In [2]:
from pathlib import Path
import requests

# Create the raw data directory
raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Official UCI dataset URLs
urls = {
    "adult.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "adult.test": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"
}

# Download each file
for filename, url in urls.items():
    file_path = raw_dir / filename
    
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    file_path.write_bytes(response.content)
    
    print(f"Downloaded: {filename}")

print("\nRaw data files downloaded successfully!")

Downloaded: adult.data
Downloaded: adult.test

Raw data files downloaded successfully!


## 4. Raw Data Inspection

Before cleaning the dataset, we inspect the raw files to understand their
structure, dimensions, column values, data types, missing-value representation,
and potential duplicate records.

The UCI Adult dataset uses `?` to represent missing categorical values.

In [3]:
# Define column names from the UCI Adult dataset
columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

# Load the raw training and test files
train_df = pd.read_csv(
    "../data/raw/adult.data",
    names=columns,
    skipinitialspace=True
)

test_df = pd.read_csv(
    "../data/raw/adult.test",
    names=columns,
    skipinitialspace=True,
    skiprows=1
)


print("Training records:", len(train_df))
print("Test records:", len(test_df))
print("Total records:", len(train_df) + len(test_df))

Training records: 32561
Test records: 16281
Total records: 48842


In [4]:
train_df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Total records:", len(train_df) + len(test_df))

Training shape: (32561, 15)
Test shape: (16281, 15)
Total records: 48842


In [6]:
test_df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K.
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K.
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K.
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K.
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K.


In [7]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education_num   32561 non-null  int64
 5   marital_status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital_gain    32561 non-null  int64
 11  capital_loss    32561 non-null  int64
 12  hours_per_week  32561 non-null  int64
 13  native_country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


### 4.1 Missing-Value Marker Inspection

The raw UCI Adult dataset represents missing categorical values using the
`?` character. Because pandas reads `?` as ordinary text, these values are
not automatically counted as missing values.

We therefore inspect the frequency of `?` before converting it to a
standard missing-value representation.

In [8]:
# Count '?' values in each column

missing_marker_counts = (train_df == "?").sum()

missing_marker_counts[missing_marker_counts > 0]

workclass         1836
occupation        1843
native_country     583
dtype: int64

In [9]:
missing_marker_counts_test = (test_df == "?").sum()

missing_marker_counts_test[missing_marker_counts_test > 0]

workclass         963
occupation        966
native_country    274
dtype: int64

### 4.2 Duplicate Audit

Exact duplicate rows are counted to understand the raw data quality.

Duplicates are not automatically removed because the dataset does not contain
a unique person identifier. Therefore, identical records cannot be assumed to
represent accidental duplicate observations.

In [10]:
# Count exact duplicate rows

print("Training duplicate rows:", train_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

Training duplicate rows: 24
Test duplicate rows: 5


## 5. Combine Raw Dataset Partitions

The UCI Adult dataset is provided as separate training and test files.

For this descriptive analysis, the two partitions are combined into a single
analysis dataset. A `source_split` column is retained to preserve the origin
of each record and maintain data traceability.

No cleaning or modification is performed during this step.

In [11]:
# Add source information before combining the datasets

train_df["source_split"] = "train"
test_df["source_split"] = "test"

# Combine the two raw datasets
adult_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

print("Combined dataset shape:", adult_df.shape)

Combined dataset shape: (48842, 16)


In [12]:
adult_df["source_split"].value_counts()

source_split
train    32561
test     16281
Name: count, dtype: int64

In [13]:
adult_df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source_split
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,train
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,train
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,train
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,train
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,train


# 6. Data Cleaning

The raw dataset contains formatting inconsistencies and missing-value markers
that need to be addressed before exploratory analysis.

The cleaning process includes:

- Standardizing column names
- Removing unnecessary whitespace
- Converting `?` markers to missing values
- Handling missing categorical values
- Normalizing income labels
- Validating numerical columns
- Checking for impossible values
- Creating analysis-ready features

## 6.1 Standardize Column Names

Column names are standardized to lowercase `snake_case` to make them
consistent and easier to work with in Python.

In [14]:
# Standardize column names

adult_df.columns = (
    adult_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

adult_df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education_num',
       'marital_status', 'occupation', 'relationship', 'race', 'sex',
       'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
       'income', 'source_split'],
      dtype='str')

### 6.2 Standardize Text Values

Leading and trailing whitespace is removed from all text columns.

This prevents values such as `" Private"` and `"Private"` from being treated
as different categories.

In [15]:
# Remove leading and trailing whitespace from all text columns

text_columns = adult_df.select_dtypes(include=["str"]).columns

adult_df[text_columns] = adult_df[text_columns].apply(
    lambda col: col.str.strip()
)

print("Whitespace standardization completed.")

Whitespace standardization completed.


### 6.3 Convert Missing-Value Markers

The UCI Adult dataset uses `?` to represent missing categorical values.

These markers are converted to pandas `NaN` so that missing values can be
identified and handled consistently.

In [16]:
# Convert '?' markers to standard missing values

adult_df = adult_df.replace("?", np.nan)

print("Missing-value markers converted to NaN.")

Missing-value markers converted to NaN.


In [17]:
adult_df.isna().sum().sort_values(ascending=False)

occupation        2809
workclass         2799
native_country     857
fnlwgt               0
education            0
education_num        0
marital_status       0
age                  0
relationship         0
race                 0
capital_gain         0
sex                  0
capital_loss         0
hours_per_week       0
income               0
source_split         0
dtype: int64

## 6.4 Handle Missing Categorical Values

Missing values are present in `workclass`, `occupation`, and `native_country`.

Instead of removing these records, missing categorical values are replaced
with `Unknown`. This preserves the observations while making the missingness
explicit for subsequent analysis.

In [18]:
# Handle missing categorical values

categorical_missing_columns = [
    "workclass",
    "occupation",
    "native_country"
]

adult_df[categorical_missing_columns] = (
    adult_df[categorical_missing_columns]
    .fillna("Unknown")
)

print("Missing categorical values replaced with 'Unknown'.")

Missing categorical values replaced with 'Unknown'.


In [19]:
adult_df[categorical_missing_columns].isna().sum()

workclass         0
occupation        0
native_country    0
dtype: int64

## 6.5 Normalize Income Labels

The training and test files use slightly different representations of the
income labels. The test file contains a trailing period (`.`).

The trailing period is removed so that both partitions use the same two
categories:

- `<=50K`
- `>50K`

In [20]:
# Remove trailing periods and standardize income labels

adult_df["income"] = (
    adult_df["income"]
    .str.strip()
    .str.rstrip(".")
)

print(adult_df["income"].value_counts())

income
<=50K    37155
>50K     11687
Name: count, dtype: int64


## 6.6 Create High-Income Flag

A binary `high_income` feature is created from the cleaned `income` label.

- `0` → income is `<=50K`
- `1` → income is `>50K`

This numerical representation will make it easier to calculate proportions
and compare income rates across different groups.

In [21]:
# Create binary high-income flag

adult_df["high_income"] = (
    adult_df["income"]
    .eq(">50K")
    .astype(int)
)

print(adult_df["high_income"].value_counts())

high_income
0    37155
1    11687
Name: count, dtype: int64


In [22]:
adult_df[["income", "high_income"]].drop_duplicates().sort_values("income")

,income,high_income
0,<=50K,0
7,>50K,1


In [23]:
adult_df["income"].value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

In [24]:
adult_df["high_income"].value_counts()

high_income
0    37155
1    11687
Name: count, dtype: int64

## 6.7 Validate Numeric Columns

The dataset contains several numerical variables, including age, census
weight, education level, capital gains, capital losses, and weekly working
hours.

These columns are checked to ensure they contain valid numeric values and
that no unexpected missing values were introduced during cleaning.

In [25]:
# Define expected numeric columns

numeric_columns = [
    "age",
    "fnlwgt",
    "education_num",
    "capital_gain",
    "capital_loss",
    "hours_per_week"
]

# Check data types
adult_df[numeric_columns].dtypes

age               int64
fnlwgt            int64
education_num     int64
capital_gain      int64
capital_loss      int64
hours_per_week    int64
dtype: object

In [26]:
adult_df[numeric_columns].isna().sum()

age               0
fnlwgt            0
education_num     0
capital_gain      0
capital_loss      0
hours_per_week    0
dtype: int64

## 6.8 Validate Core Numeric Ranges

Basic range checks are performed to identify impossible or invalid values in
the numerical variables.

Rows with invalid core values would be reviewed before analysis rather than
being removed automatically.

In [27]:
# Check for invalid numeric values

invalid_checks = {
    "age <= 0": (adult_df["age"] <= 0).sum(),
    "education_num <= 0": (adult_df["education_num"] <= 0).sum(),
    "hours_per_week <= 0": (adult_df["hours_per_week"] <= 0).sum(),
    "capital_gain < 0": (adult_df["capital_gain"] < 0).sum(),
    "capital_loss < 0": (adult_df["capital_loss"] < 0).sum()
}

pd.Series(invalid_checks)

age <= 0               0
education_num <= 0     0
hours_per_week <= 0    0
capital_gain < 0       0
capital_loss < 0       0
dtype: int64

## 7. Feature Engineering

Feature engineering involves creating additional variables from existing
data to support meaningful analysis.

For this project, we create age groups, weekly working-hour groups, and a
net capital measure while retaining the original variables.

### 7.1 Create Age Groups

Age is grouped into meaningful ranges to support comparison of income patterns
across different stages of working life.

The first group begins at age 17 because the minimum age in the dataset is 17.

In [28]:
age_bins = [16, 25, 35, 45, 55, 65, np.inf]
age_labels = ["17-25", "26-35", "36-45", "46-55", "56-65", "66+"]

adult_df["age_group"] = pd.cut(
    adult_df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

adult_df["age_group"].value_counts().sort_index()

age_group
17-25     9627
26-35    12719
36-45    11952
46-55     8296
56-65     4445
66+       1803
Name: count, dtype: int64

### 7.2 Create Weekly Working-Hour Groups

Weekly working hours are grouped into ranges to examine whether income
patterns differ across typical and higher/lower working-hour categories.

In [29]:
# Create weekly working-hour groups

hours_bins = [0, 20, 34, 40, 49, 59, np.inf]
hours_labels = [
    "<=20",
    "21-34",
    "35-40",
    "41-49",
    "50-59",
    "60+"
]

adult_df["hours_group"] = pd.cut(
    adult_df["hours_per_week"],
    bins=hours_bins,
    labels=hours_labels,
    include_lowest=True
)

adult_df["hours_group"].value_counts().sort_index()

hours_group
<=20      4453
21-34     3942
35-40    26095
41-49     4671
50-59     5828
60+       3853
Name: count, dtype: int64

### 7.3 Create Net Capital

A `net_capital` feature is calculated as capital gain minus capital loss.

This provides a single measure that summarizes the two capital-related
variables while retaining the original `capital_gain` and `capital_loss`
columns.

In [30]:
# Calculate net capital

adult_df["net_capital"] = (
    adult_df["capital_gain"] - adult_df["capital_loss"]
)

adult_df["net_capital"].describe()

count    48842.000000
mean       991.565313
std       7475.549906
min      -4356.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      99999.000000
Name: net_capital, dtype: float64

### 7.4 Verify Engineered Features

The newly created features are reviewed alongside the original variables to
confirm that the feature engineering process has been applied correctly.

The original variables are retained so that the derived features can be
validated and compared during the analysis.

In [31]:
# Preview the engineered features

adult_df[
    [
        "age",
        "age_group",
        "hours_per_week",
        "hours_group",
        "capital_gain",
        "capital_loss",
        "net_capital"
    ]
].head()

,age,age_group,hours_per_week,hours_group,capital_gain,capital_loss,net_capital
0,39,36-45,40,35-40,2174,0,2174
1,50,46-55,13,<=20,0,0,0
2,38,36-45,40,35-40,0,0,0
3,53,46-55,40,35-40,0,0,0
4,28,26-35,40,35-40,0,0,0


## 8. Final Data Quality Audit

A final quality audit is performed after cleaning and feature engineering
to verify that the dataset is complete, consistent, and ready for analysis.

In [32]:
# Final data quality audit

print("Final dataset shape:", adult_df.shape)

print("\nMissing values:")
print(adult_df.isna().sum().sum())

print("\nDuplicate rows:")
print(adult_df.duplicated().sum())

print("\nIncome distribution:")
print(adult_df["income"].value_counts())

print("\nSource split:")
print(adult_df["source_split"].value_counts())

print("\nFinal columns:")
print(adult_df.columns.tolist())

Final dataset shape: (48842, 20)

Missing values:
0

Duplicate rows:
29

Income distribution:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Source split:
source_split
train    32561
test     16281
Name: count, dtype: int64

Final columns:
['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income', 'source_split', 'high_income', 'age_group', 'hours_group', 'net_capital']


## 9. Export Cleaned Dataset

The cleaned and feature-engineered dataset is exported as a CSV file so
that it can be reused for exploratory analysis and future predictive
modeling without repeating the data-cleaning steps.

In [33]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "adult_income_cleaned.csv"

adult_df.to_csv(output_path, index=False)

print(f"Cleaned dataset exported to: {output_path}")
print(f"Final dataset shape: {adult_df.shape}")

Cleaned dataset exported to: ..\data\processed\adult_income_cleaned.csv
Final dataset shape: (48842, 20)


In [34]:
# Verify exported file

processed_file = Path("../data/processed/adult_income_cleaned.csv")

print("File exists:", processed_file.exists())

if processed_file.exists():
    print("File size (KB):", round(processed_file.stat().st_size / 1024, 2))

File exists: True
File size (KB): 6279.23
